In [2]:
import pandas as pd
import numpy as np
import os

# Load
orders_clean = pd.read_csv('orders_clean.csv')
order_behavior = pd.read_csv('order_behavior_base.csv')
labels = pd.read_csv('data/processed/instacart/labels/instacart_phase2_decay_labels.csv')

orders_clean = orders_clean.sort_values(['user_id', 'order_number']).copy()

# Relative day / tenure
orders_clean['relative_day'] = (
    orders_clean.groupby('user_id')['days_since_prior_order']
    .apply(lambda s: s.fillna(0).cumsum())
    .reset_index(level=0, drop=True)
)
orders_clean['base_user_tenure_days'] = orders_clean['relative_day']
orders_clean['base_order_number'] = orders_clean['order_number']
orders_clean['base_total_orders_to_date'] = orders_clean['order_number']

# Expanding cadence
orders_clean['base_avg_days_between_orders'] = (
    orders_clean.groupby('user_id')['days_since_prior_order']
    .expanding(min_periods=1).mean()
    .reset_index(level=0, drop=True)
)

# Basket size / reorder ratio
orders_clean = orders_clean.merge(
    order_behavior[['order_id', 'item_count', 'reorder_ratio']], on='order_id', how='left'
)
orders_clean['base_avg_basket_size_to_date'] = (
    orders_clean.groupby('user_id')['item_count']
    .expanding(min_periods=1).mean()
    .reset_index(level=0, drop=True)
)
orders_clean['base_avg_reorder_ratio_to_date'] = (
    orders_clean.groupby('user_id')['reorder_ratio']
    .expanding(min_periods=1).mean()
    .reset_index(level=0, drop=True)
)

# Day-of-week / hour
orders_clean['base_order_dow'] = orders_clean['order_dow']
orders_clean['base_order_hour'] = orders_clean['order_hour_of_day']

# Assemble final output
baseline_features = orders_clean[[
    'user_id', 'order_id', 'base_user_tenure_days', 'base_order_number',
    'base_total_orders_to_date', 'base_avg_days_between_orders',
    'base_avg_basket_size_to_date', 'base_avg_reorder_ratio_to_date',
    'base_order_dow', 'base_order_hour'
]].copy()

eligible_keys = labels[['user_id', 'order_id']]
baseline_features = baseline_features.merge(eligible_keys, on=['user_id', 'order_id'], how='inner')

# Verification checks
assert len(baseline_features) == len(labels), "Row count mismatch with label file!"
assert baseline_features.duplicated(['user_id', 'order_id']).sum() == 0, "Duplicate keys found!"

forbidden = ['next_order_id','next_order_number','next_eval_set','next_gap_days',
             'next_gap_ratio_to_historical_median','early_decay_label','label_eligible',
             'next_gap_30d_label','gap_cap_30_flag','next_gap_1_5x_median_label',
             'next_gap_2x_median_label','next_gap_2_5x_median_label',
             'decay_severity_score','decay_severity_tier','label_definition']
assert not any(c in baseline_features.columns for c in forbidden), "Leaked column detected!"

# Save
os.makedirs('data/processed/instacart/features', exist_ok=True)
baseline_features.to_csv('data/processed/instacart/features/baseline_features.csv', index=False)

print(f"Saved {len(baseline_features):,} rows to baseline_features.csv")

Saved 2,593,914 rows to baseline_features.csv


In [3]:
import pandas as pd
check = pd.read_csv('data/processed/instacart/features/baseline_features.csv')
print(len(check))
print(check.duplicated(['user_id','order_id']).sum())
print(check.columns.tolist())

2593914
0
['user_id', 'order_id', 'base_user_tenure_days', 'base_order_number', 'base_total_orders_to_date', 'base_avg_days_between_orders', 'base_avg_basket_size_to_date', 'base_avg_reorder_ratio_to_date', 'base_order_dow', 'base_order_hour']
